# 03 Evaluate PR Suggestion Metrics

Purpose: generate `reports/metric_scores.csv` and print baseline-vs-current metric quality.

This notebook uses the reusable evaluator in `src/pr_suggestion_metrics/evaluate_metrics.py`.

The evaluator is language-aware for token metrics. It selects tokenization from the changed file path: Python, Jupyter Notebook code cells, JSON, YAML, Markdown, Dockerfile, Pygments for broad language coverage such as Go/C++/Rust/Java/TypeScript/JavaScript/C/HTML/Groovy/HCL/Shell, or a generic regex fallback when no lexer is available.

The evaluator also computes literal-normalized recall, so small value changes like `30 -> 60` or `"dev" -> "prod"` reduce the score less than unrelated code.

The evaluator also emits optional structural evidence. Python and Jupyter code cells use local stdlib `ast` scoring. Go/C++/Rust/Java/TypeScript/JavaScript/C/HTML use the locked `tree-sitter-language-pack` dependency. GumTree/code-diff remains separate and opt-in.

`ENABLE_GUMTREE = False` by default because GumTree/code-diff is parser-dependent and may try to fetch Tree-sitter parsers. Set it to `True` only when you want optional GumTree edit-script diagnostics.


In [ ]:
from __future__ import annotations

import csv
import subprocess
import sys
from collections import Counter
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
SRC_PATH = PROJECT_ROOT / 'src'
DATASET_DIR = PROJECT_ROOT / 'data' / 'processed' / 'pr_suggestion_coverage' / 'dataset'
OUTPUT_PATH = PROJECT_ROOT / 'reports' / 'metric_scores.csv'
EVALUATOR_PATH = PROJECT_ROOT / 'src' / 'pr_suggestion_metrics' / 'evaluate_metrics.py'
ENABLE_GUMTREE = False

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
DATASET_DIR, OUTPUT_PATH, ENABLE_GUMTREE


## Run Evaluator

In [ ]:
cmd = [
    sys.executable,
    str(EVALUATOR_PATH),
    '--dataset-dir',
    str(DATASET_DIR),
    '--output',
    str(OUTPUT_PATH),
]
if ENABLE_GUMTREE:
    cmd.append('--enable-gumtree')

print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
OUTPUT_PATH

## Inspect Scores

In [ ]:
with OUTPUT_PATH.open(newline='', encoding='utf-8') as scores_file:
    rows = list(csv.DictReader(scores_file))

print('rows:', len(rows))
print('actual labels:', Counter(row['label'] for row in rows))
print('predicted labels:', Counter(row['predicted_label'] for row in rows))
if 'suggestion_language' in rows[0]:
    print('suggestion languages:', Counter(row['suggestion_language'] for row in rows))
if 'tokenizer' in rows[0]:
    print('tokenizers:', Counter(row['tokenizer'] for row in rows))
if 'best_hunk_candidate_type' in rows[0]:
    print('best hunk candidate types:', Counter(row['best_hunk_candidate_type'] or 'none' for row in rows))
if 'structural_available' in rows[0]:
    print('structural availability:', Counter(row['structural_available'] for row in rows))
    print('structural engines:', Counter(row['structural_engine'] or 'none' for row in rows))
if 'gumtree_available' in rows[0]:
    print('gumtree availability:', Counter(row['gumtree_available'] for row in rows))

rows[:5]


## Literal Normalization Impact

This checks whether literal-tolerant metrics are active in the generated score file.


In [ ]:
def as_float(row: dict[str, str], column: str) -> float:
    return float(row.get(column) or 0.0)

literal_metric_pairs = [
    ('token_recall', 'literal_normalized_token_recall'),
    ('identifier_normalized_token_recall', 'identifier_and_literal_normalized_token_recall'),
    ('best_hunk_token_recall', 'best_hunk_literal_normalized_recall'),
    ('best_hunk_identifier_normalized_recall', 'best_hunk_identifier_and_literal_normalized_recall'),
]

for base_column, literal_column in literal_metric_pairs:
    if base_column not in rows[0] or literal_column not in rows[0]:
        print(f'missing columns: {base_column}, {literal_column}')
        continue
    base_mean = sum(as_float(row, base_column) for row in rows) / len(rows)
    literal_mean = sum(as_float(row, literal_column) for row in rows) / len(rows)
    print(f'{base_column} mean: {base_mean:.3f}')
    print(f'{literal_column} mean: {literal_mean:.3f}')
    print(f'delta: {literal_mean - base_mean:+.3f}')
    print()



## Single Example Inspector

Set `SELECTED_EXAMPLE_ID` to inspect one row with the same evaluator functions used by the script.


In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location('evaluate_metrics', EVALUATOR_PATH)
evaluator = importlib.util.module_from_spec(spec)
assert spec.loader is not None
sys.modules['evaluate_metrics'] = evaluator
spec.loader.exec_module(evaluator)

examples = evaluator._load_labeled_examples(DATASET_DIR)
examples_by_id = {example.example_id: example for example in examples}
scores_by_id = {row['example_id']: row for row in rows}

SELECTED_EXAMPLE_ID = rows[0]['example_id']
selected_example = examples_by_id[SELECTED_EXAMPLE_ID]
selected_score = scores_by_id[SELECTED_EXAMPLE_ID]

print('selected example:', SELECTED_EXAMPLE_ID)
print('label:', selected_example.label)
print('expected:', selected_example.expected_landed_percentage)
print('predicted:', selected_score['predicted_percentage'], selected_score['predicted_label'])
print('language/tokenizer:', selected_score.get('suggestion_language'), selected_score.get('tokenizer'))
print('best hunk:', selected_score.get('best_hunk_file'), 'type=', selected_score.get('best_hunk_candidate_type'), 'size=', selected_score.get('best_hunk_size'))
print('candidate hunks checked:', selected_score.get('candidate_hunk_count'))

{
    key: selected_score[key]
    for key in selected_score
    if key.endswith('_recall')
    or key.endswith('_precision')
    or key.endswith('_f1')
    or key.endswith('_ratio')
    or key.startswith('structural_')
}


## Token And Hunk Debugger

This shows the exact per-file tokenization, normalized token samples, best-hunk scores, and structural scores.


In [ ]:
suggested_lines_by_file = evaluator._prepare_lines_by_file(
    evaluator._added_lines_by_file_from_diff(selected_example.suggested_diff)
)
landed_lines_by_file = evaluator._prepare_lines_by_file(
    evaluator._added_lines_by_file_from_diff(selected_example.landed_diff)
)
landed_hunks_by_file = evaluator._prepare_hunks_by_file(
    evaluator._added_hunks_by_file_from_diff(selected_example.landed_diff)
)

for path, suggested_lines in suggested_lines_by_file.items():
    text = '\n'.join(suggested_lines)
    tokenized = evaluator._tokenize_for_path(path, text)
    identifier_tokens = evaluator._normalize_identifier_tokens(tokenized.tokens)
    literal_tokens = evaluator._normalize_literal_tokens(tokenized.tokens)
    both_tokens = evaluator._normalize_identifier_and_literal_tokens(tokenized.tokens)

    print('file:', path)
    print('language:', tokenized.language)
    print('tokenizer:', tokenized.tokenizer)
    print('suggested lines:', len(suggested_lines))
    print('landed lines in same file:', len(landed_lines_by_file.get(path, [])))
    print('tokens:', tokenized.tokens[:80])
    print('identifier-normalized:', identifier_tokens[:80])
    print('literal-normalized:', literal_tokens[:80])
    print('identifier+literal-normalized:', both_tokens[:80])
    print()

best_hunk_scores = evaluator._best_hunk_scores(suggested_lines_by_file, landed_hunks_by_file)
structural_scores = evaluator._best_structural_scores(suggested_lines_by_file, landed_hunks_by_file)

print('best hunk scores')
for key, value in best_hunk_scores.items():
    print(f'{key}: {value}')

print()
print('structural scores')
for key, value in structural_scores.items():
    print(f'{key}: {value}')


## Diff Preview

Use this to read the raw suggestion and landed diff for the selected example.


In [ ]:
MAX_DIFF_CHARS = 4000

print('SUGGESTED DIFF')
print(selected_example.suggested_diff[:MAX_DIFF_CHARS])
if len(selected_example.suggested_diff) > MAX_DIFF_CHARS:
    print('... truncated ...')

print('\nLANDED DIFF')
print(selected_example.landed_diff[:MAX_DIFF_CHARS])
if len(selected_example.landed_diff) > MAX_DIFF_CHARS:
    print('... truncated ...')


## Structural Backend Smoke Test

This checks parser availability for P0 plus the P1 structural candidates using the same local helper available from the command line. It is independent of whether the current dataset contains examples for each language.


In [ ]:
from pr_suggestion_metrics.prepare_structural_parsers import check_structural_parsers

for check in check_structural_parsers(['python', 'go', 'cpp', 'rust', 'java', 'typescript', 'javascript', 'c', 'html']):
    print(check.language, 'engine=', check.engine, 'nodes=', check.node_count, 'error=', check.error or 'none')



## Structural Notes

Python structural scoring is always local and uses the standard-library `ast` module.

Go/C++/Rust/Java/TypeScript/JavaScript/C/HTML structural scoring uses the locked `tree-sitter-language-pack` dependency. Run `uv run --locked --extra structural pr-suggestion-prepare-parsers` from the repository root to verify it outside the notebook.

Jupyter Notebook files are compared by extracted code-cell source instead of raw `.ipynb` JSON. Groovy, HCL, and Shell currently use language-aware tokenization; structural scoring remains disabled for them until real examples prove parser stability on partial snippets.

GumTree is separate. It is useful for Type-3 clone detection, but it depends on parser support and local parser availability.

Current behavior:

- token metrics detect language from file extension or Dockerfile-like filename
- unsupported file types use generic regex tokens
- Python AST and local Tree-sitter structural scoring record `structural_*` columns
- GumTree is opt-in with `ENABLE_GUMTREE = True`
- GumTree only runs for supported source languages
- unsupported/parser failures are recorded in `gumtree_available` and `gumtree_error`
